# SMART TIME — Free GPU smoke training

This notebook is designed for a free Kaggle Notebook GPU session. Start with the 1.7B smoke run; do not use customer data or Voice DNA recordings.

In [ ]:
!git clone -b feat/smart-voice-dna-local https://github.com/msm147852/SMART-TIME-.git
%cd SMART-TIME-
!git rev-parse --short HEAD

In [ ]:
!nvidia-smi
!python infra/smart-ai/training/preflight.py

In [ ]:
!pip install -q -r infra/smart-ai/training/requirements.txt
!python -m pip check

In [ ]:
!python - <<'PY'
import json
from pathlib import Path
files = [Path('backend/ai/training/smart-time-v2.jsonl'), Path('backend/ai/training/smart-time-grounded-v1.jsonl')]
rows=[]
for f in files:
    rows += [json.loads(line) for line in f.read_text(encoding='utf-8').splitlines() if line.strip()]
print('training examples:', len(rows))
assert len(rows) >= 90
eval_rows=[json.loads(line) for line in Path('backend/ai/training/smart-time-eval-v1.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
print('held-out eval:', len(eval_rows))
assert len(eval_rows) >= 20
PY

In [ ]:
!mkdir -p /kaggle/working/smart-ai-output
BASE_MODEL=Qwen/Qwen3-1.7B \
OUTPUT_DIR=/kaggle/working/smart-ai-output \
EPOCHS=1 \
BATCH_SIZE=2 \
GRADIENT_ACCUMULATION_STEPS=4 \
USE_LORA=1 \
python infra/smart-ai/training/train_smart_ai.py

In [ ]:
MODEL=/kaggle/working/smart-ai-output \
BASE_MODEL=Qwen/Qwen3-1.7B \
EVAL_OUTPUT=/kaggle/working/eval-predictions.jsonl \
python infra/smart-ai/training/evaluate_smart_ai.py
!python scripts/score-smart-ai-eval.mjs /kaggle/working/eval-predictions.jsonl